# M1 — LoRA fine-tuning of Qwen3 for "MacGyver Gym Rat"

**SI4006 · Entrega M1 — Fine-tuning baseline del proyecto**

This notebook teaches a small language model to answer one question honestly:

> *"I want to train my delts but I have no gym equipment. All I have is two filled water bottles."*

It must name a **real exercise from a 1324-entry catalog**, state the gym equipment it
stands in for, explain the substitution, and recite **that exercise's real instructions** —
or say plainly that nothing in the catalog works, instead of inventing something.

The failure it exists to prevent: an untuned model asked this question invents exercises and
invents biomechanics. In a domain where the cost of a wrong answer is an injury, a confident
hallucination is worse than a refusal.

Sections:

1. Setup
2. Base model and tokenizer — and why this model
3. Dataset — how a gym catalog becomes a "survival fitness" dataset
4. Baseline — the same model, *without* fine-tuning
5. LoRA configuration and training
6. Final evaluation against the baseline
7. Qualitative examples
8. Honest reading

Everything is seeded (`seed = 42`). Cells are meant to be run in order, top to bottom.

## 1 · Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/iamcroody/models-for-exercises-dataset.git"
BRANCH = "FinetuningEntregaM1"
IN_COLAB = "google.colab" in sys.modules

# Switch to "Qwen/Qwen3-4B-Instruct-2507" with FOUR_BIT = True for the larger run.
MODEL = "Qwen/Qwen3-1.7B"
FOUR_BIT = False

if IN_COLAB:
    if not Path("models-for-exercises-dataset").exists():
        subprocess.run(["git", "clone", "--recurse-submodules", "-b", BRANCH, REPO],
                       check=True)
    os.chdir("models-for-exercises-dataset")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "trl", "peft", "datasets", "rouge-score",
                    "bitsandbytes"], check=True)
    # Colab ships torchao 0.10 pinned to its torch build; peft refuses anything
    # below 0.16, and upgrading it drags a different torch in behind it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=False)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path("scripts").resolve()))
QUANT = ["--4bit"] if FOUR_BIT else []
print("working directory:", Path.cwd())
print("model:", MODEL, "| 4-bit:", FOUR_BIT)

In [ ]:
import torch
import macgyver as mg

device, DTYPE = mg.pick_device_dtype()
print(f"torch   {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    major, minor = torch.cuda.get_device_capability()
    print(f"gpu     {props.name} ({props.total_memory / 1e9:.1f} GB, sm_{major}{minor})")
print(f"dtype   {DTYPE}")
print()
print("Precision is read from the device, not hard-coded. Colab's free T4 is Turing")
print("(sm_75) and has no bf16 units, while TRL's SFTConfig defaults bf16=True whenever")
print("fp16 is unset. torch.cuda.is_bf16_supported() is a trap here: it also counts")
print("bf16 that torch emulates in software, which runs slowly and never errors, so")
print("mg.pick_device_dtype asks for compute capability instead.")

## 2 · Base model and tokenizer

**Family: decoder.** The task is open-ended generation — a recommendation, a substitution and
a set of instructions — not a label from a closed set. An encoder classifier cannot emit
prose, and an encoder-decoder buys nothing here because the input is a short question rather
than a document to transform. This model also carries into M2 (RAG on top of it) and M3
(a visual component), which an encoder classifier could not host.

**Model: `Qwen/Qwen3-1.7B`.**

- **Apache-2.0 and ungated.** No Hugging Face login, no licence click-through. Llama-3.2 is
  gated, which would break "anyone can open this notebook and run it".
- **Fits the free tier with room.** ~1.0% of parameters trainable under LoRA, ~5.8 GB peak
  against a T4's 15.6 GB — measured, not estimated.
- **Has a forward path.** `Qwen3-VL-2B` shares this tokenizer and chat template, so M3's
  visual component is a family swap rather than a rewrite, and the catalog already ships a
  thumbnail and an animation for all 1324 exercises.

`Qwen/Qwen3-4B-Instruct-2507` in 4-bit also fits (5.6 GB peak, 17.8 s/step against 6.7).
Set `FOUR_BIT = True` above to run it.

In [ ]:
# Week 3 Lab A advice: before committing to a base model, look at how it tokenizes the
# vocabulary the task turns on. Here that is two things the model must reproduce exactly —
# catalog exercise names, and the household-object phrases in the prompt.
from transformers import AutoTokenizer

catalog = mg.load_catalog()
names = [r["name"] for r in catalog]
objects = [o for eq in mg.OBJECT_MAP for o in mg.objects_for(eq, include_holdout=True)]

print(f"{'tokenizer':<34} {'vocab':>7} {'names':>7} {'objects':>8}")
print("-" * 60)
for model_id in [MODEL, "HuggingFaceTB/SmolLM2-1.7B-Instruct", "openai-community/gpt2"]:
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
    except Exception as exc:
        print(f"{model_id:<34} unavailable ({type(exc).__name__})")
        continue
    def per_word(texts):
        return sum(len(tok.encode(t, add_special_tokens=False)) for t in texts) / sum(
            len(t.split()) for t in texts)
    print(f"{model_id:<34} {len(tok):>7} {per_word(names):>7.2f} {per_word(objects):>8.2f}")
print()
print("(tokens per word — lower is better)")

## 3 · Dataset

1324 exercises from a git submodule pinned to a fixed commit, so every run trains on
byte-identical data. Full description, licence and known biases: `docs/DATASET.md`.

**The catalog contains no household objects at all.** So the dataset is built by crossing it
with a hand-written equivalence table — four equipment classes, 21 object phrases, four
adaptation sentences. Everything else is real:

| piece | source |
|---|---|
| exercise name, equipment, target, every instruction step | **catalog, verbatim** |
| object equivalences, prompt wording, adaptation and safety lines | this repo's script, deterministic |
| anything written by a language model | **none** |

That last row matters. The course's data guide warns that synthetic data inherits the
generator's style and biases, and collapses into near-duplicates. Neither can happen here:
no model is in the loop, and the diversity comes from 1324 real records rather than repeated
sampling of one prompt.

In [ ]:
!python scripts/01_macgyver_data.py

### Three decisions in how this dataset is built

**Answers may need nothing beyond the object the user offered.** The catalog's `equipment`
field records only the *primary* implement, so `exercise ball supine triceps extension` is
filed under `dumbbell` while step 1 asks for an exercise ball, and `body weight` means "no
external load" rather than "no equipment" — 111 of its 325 records still want a pull-up bar.
Telling someone holding two bricks to lie on a bench is not adaptation. Enforcing this costs
338 of 696 mapped exercises and leaves 358.

All three defects behind that rule were found by **reading generated examples**, not by
reasoning about the schema. The filter is keyword-based, so more probably survive.

**The split is by exercise ID, stratified on target.** If an exercise lands in validation the
model never saw its steps during training, so it cannot recite them from memory — it has to
generalise the structure.

**Two objects per equipment class never appear in training.** `two bricks`,
`a crate of bottled water`, `a patch of grass in the park` and five others are validation-only.
With a handful of phrases a model can memorise `water bottles → dumbbell` as a lookup instead
of learning `a matched pair you can grip → dumbbell`. Holding objects out turns that risk into
a measured number.

In [ ]:
meta = mg.load_meta()
train_rows, val_rows = mg.load_split("train"), mg.load_split("val")

print("counts:", json.dumps(meta["counts"], indent=2) if False else meta["counts"])
print("held-out objects:", [o for e in mg.OBJECT_MAP.values() for o in e["holdout"]])
print()
print("=== one training example, exactly as the model sees it ===")
print(train_rows[0]["prompt"][0]["content"])
print()
print("--- expected completion ---")
print(train_rows[0]["completion"][0]["content"])

In [ ]:
# Leakage checks, run rather than asserted.
held = {o for e in mg.OBJECT_MAP.values() for o in e["holdout"]}
shared = ({r["exercise_id"] for r in train_rows} & {r["exercise_id"] for r in val_rows}) - {None}
print("held-out objects appearing in train:", sum(r["object"] in held for r in train_rows))
print("exercise ids in both train and val: ", len(shared))
print("validation examples on unseen objects:", sum(not r["object_seen"] for r in val_rows))

### The scorer, and why it does not compare against the gold answer

A request for `delts` with water bottles has dozens of correct answers; our generator picked
one. Grading against that single string would mark most correct answers wrong.

So every check **resolves the exercise the model named in the real catalog** and asks whether
that choice satisfies the request:

- **Constraint satisfaction** (headline) — the exercise exists, its target is the one asked
  for, and its equipment is the class the offered object stands in for.
- **Step grounding (ROUGE-L)** — the recited steps against *that named exercise's* real steps.
  High means the biomechanics came from the catalog; low means they were invented. It is
  `None`, not `0`, when the exercise does not exist: there is no reference, and averaging a
  zero in would blend two different failures.
- **Refusal precision and recall** — one without the other is gameable. A model that refuses
  everything gets perfect recall; one that never refuses gets perfect precision.
- **Seen vs unseen objects** — a gap means memorised phrases rather than learned roles.

This also keeps the evaluation honest under the data guide's rule that a test set must be
real. Our completions are templated; the catalog is not, and it is the catalog that grades.

Before trusting any number, the scorer grades the gold answers. They are correct by
construction, so every metric must come out perfect — if it does not, the scorer is wrong and
every later number would inherit the error silently.

In [ ]:
!python scripts/02_score.py --split val

## 4 · Baseline — the same model, without fine-tuning

The assignment recommends this baseline and it is the only one that isolates what LoRA
contributed: same prompts, same split, same greedy decoding, same scorer. The *only*
difference from section 6 is whether an adapter is loaded, so the delta cannot be an artefact
of the harness.

Greedy rather than sampled decoding, because temperature would add run-to-run variance to
numbers a reader is asked to compare across rows of a table.

In [ ]:
!python scripts/03_eval_zeroshot.py --model {MODEL} {" ".join(QUANT)} \
    --split val --report-name zeroshot

## 5 · LoRA configuration and training

No full fine-tuning: only low-rank adapters on frozen base weights.

| hyperparameter | value | why |
|---|---|---|
| `r` | 16 | 662 examples over ~358 exercises is a small, narrow target. The job is to index a closed catalog and recite it, not to install knowledge about human movement. |
| `lora_alpha` | 32 | Holds `alpha = 2r`, so the effective scale `alpha/r` stays at 2.0 and changing `r` alone does not silently change the update magnitude too. |
| `target_modules` | all 7 linear projections | The course material suggests attention only. Mapping "two filled water bottles" → `dumbbell` is lexical-semantic, and that lives largely in the MLP blocks. `--target-modules attention` tests the claim rather than asserting it. |
| `lora_dropout` | 0.05 | Light regularisation on a small dataset. |
| `learning_rate` | 1e-4 | TRL's documented adapter rate, ~5× a full fine-tune's, because only the freshly-initialised low-rank matrices learn. |
| epochs | 3 | 662 examples is small, and refusals are only 8% of train — two passes underfit them. |
| effective batch | 16 | `2 × 8`, fixed rather than scaled to the GPU, so a local run and the Colab run take identical optimisation steps. |

TRL computes the loss on the **completion only** for prompt-completion datasets
(`completion_only_loss` defaults to `True`), so the format spec in the prompt costs nothing at
training time — the model is never asked to predict it.

In [ ]:
!python scripts/04_train_lora.py --model {MODEL} {" ".join(QUANT)}

## 6 · Final evaluation, against the same validation set

Same script, same scorer, same 154 validation examples — only `--adapter` differs. One code
path means the delta cannot come from two eval implementations quietly disagreeing.

In [ ]:
ADAPTER = "models/r16-all-linear"

!python scripts/03_eval_zeroshot.py --model {MODEL} {" ".join(QUANT)} \
    --split val --adapter {ADAPTER} --report-name finetuned

In [ ]:
import json

zeroshot = json.loads(Path("reports/zeroshot.json").read_text())
finetuned = json.loads(Path("reports/finetuned.json").read_text())

rows = [
    ("format ok",                 "format_ok",               "pct"),
    ("constraint satisfaction",   "constraint_satisfaction", "pct"),
    ("  exercise is real",        "exercise_real",           "pct"),
    ("  target matches",          "target_match",            "pct"),
    ("  equipment matches",       "equipment_match",         "pct"),
    ("step grounding ROUGE-L",    "step_grounding_rouge_l",  "num"),
]

def fmt(value, kind):
    if value is None:
        return "n/a"
    return f"{value:.1%}" if kind == "pct" else f"{value:.3f}"

print(f"{'metric':<26} {'baseline':>10} {'fine-tuned':>12} {'delta':>10}")
print("-" * 62)
for label, key, kind in rows:
    a, b = zeroshot[key], finetuned[key]
    delta = fmt(b - a, kind) if (a is not None and b is not None) else "n/a"
    print(f"{label:<26} {fmt(a, kind):>10} {fmt(b, kind):>12} {delta:>10}")

print()
for label in ("seen_objects", "unseen_objects"):
    a, b = zeroshot[label], finetuned[label]
    print(f"{label.replace('_', ' '):<26} {fmt(a['constraint_satisfaction'], 'pct'):>10} "
          f"{fmt(b['constraint_satisfaction'], 'pct'):>12}   (n={b['n']})")

print()
for name, report in (("baseline", zeroshot), ("fine-tuned", finetuned)):
    r = report["refusal"]
    print(f"refusal {name:<18} P {r['precision']:.1%}  R {r['recall']:.1%}  F1 {r['f1']:.1%}")

## 7 · Qualitative examples

Three examples, picked by rule rather than by hand: the first answerable row, the first
refusable one, and the first using an object the model was never trained on.

In [ ]:
for a, b in zip(zeroshot["examples"], finetuned["examples"]):
    print("=" * 78)
    print(f"[{b['why_shown']}]  {b['prompt'].splitlines()[0]}")
    print("=" * 78)
    print("\n--- BASELINE (no fine-tuning) ---")
    print(a["generated"].strip()[:700])
    print("\n--- FINE-TUNED ---")
    print(b["generated"].strip()[:700])
    print("\n--- GOLD (one valid answer of many) ---")
    print(b["gold"].strip()[:700])
    print()

## 8 · Honest reading

Fill this in from the numbers above once the notebook has run end to end, and keep it
honest — where it still fails is as interesting as where it improved.

Points worth checking before writing it:

- **Does the seen/unseen object gap close?** If unseen-object satisfaction is far below seen,
  the model memorised phrases rather than roles, and the equivalence table needs more objects
  rather than the model needing more capacity.
- **Is refusal recall real, or bought with precision?** 46 refusable examples is enough to
  read a rate off, not enough for a tight confidence interval.
- **A lookup table would beat this.** 40 lines of Python indexing the catalog by
  `(target, equipment)` scores 100% on constraint satisfaction. What the fine-tune buys is
  that it does this in natural language, inside a model M2 will put RAG on and M3 will give
  eyes — not that it is the best way to answer this one question today.
- **154 validation examples.** One example moves the headline by 0.6 points, so differences of
  a point or two between configurations are noise and should not be read as meaningful.

The test split has deliberately not been touched. It is reserved for M2, so the rigorous
evaluation there is not run on a set already used to make decisions here.